In [21]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [22]:
import pandas as pd

df = pd.read_csv('../data/csv/combined_dataset.csv', low_memory=False)

# Label temizliği
df['Label'] = df['Label'].astype(str).str.strip()
df['Label'] = df['Label'].replace({
    'Web Attack � Brute Force': 'Web Attack - Brute Force',
    'Web Attack � XSS': 'Web Attack - XSS',
    'Web Attack � Sql Injection': 'Web Attack - Sql Injection'
})

print(f"Dataset yüklendi: {df.shape[0]:,} satır, {df.shape[1]} sütun")

Dataset yüklendi: 1,101,564 satır, 80 sütun


In [23]:
print("=== DATASET SUMMARY ===")
print(f"Satır: {df.shape[0]:,}")
print(f"Sütun: {df.shape[1]}")
print(f"\nVeri tipleri:\n{df.dtypes.value_counts()}")
print(f"\nLabel dağılımı:")
print(df['Label'].value_counts())

=== DATASET SUMMARY ===
Satır: 1,101,564
Sütun: 80

Veri tipleri:
int64      54
float64    24
object      2
Name: count, dtype: int64

Label dağılımı:
Label
BENIGN                        543918
DoS Hulk                      231073
PortScan                      158930
DDoS                          128027
DoS GoldenEye                  10293
FTP-Patator                     7938
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1966
Web Attack - Brute Force        1507
Web Attack - XSS                 652
Infiltration                      36
Web Attack - Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64


In [24]:
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("✓ Eksik değer (NaN) yok")
else:
    print(f"Eksik değer bulunan sütunlar:")
    print(missing)

Eksik değer bulunan sütunlar:
Flow Bytes/s    1014
dtype: int64


In [25]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
inf_counts = {}
for col in numeric_cols:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        inf_counts[col] = inf_count

if not inf_counts:
    print("✓ Inf değer yok")
else:
    print("Inf değer bulunan sütunlar:")
    for col, count in inf_counts.items():
        print(f"  {col}: {count:,} adet")

Inf değer bulunan sütunlar:
  Flow Bytes/s: 525 adet
  Flow Packets/s: 1,539 adet


In [26]:
before = len(df)

# Inf → NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Kaç satır NaN içeriyor?
nan_rows = df.isnull().any(axis=1).sum()
print(f"NaN içeren satır: {nan_rows:,}")

# NaN içeren satırları sil
df.dropna(inplace=True)

after = len(df)

print(f"\nTemizleme öncesi: {before:,} satır")
print(f"Temizleme sonrası: {after:,} satır")
print(f"Silinen satır: {before - after:,} ({(before-after)/before*100:.2f}%)")

print(f"\nLabel dağılımı (temizleme sonrası):")
print(df['Label'].value_counts())

NaN içeren satır: 1,539

Temizleme öncesi: 1,101,564 satır
Temizleme sonrası: 1,100,025 satır
Silinen satır: 1,539 (0.14%)

Label dağılımı (temizleme sonrası):
Label
BENIGN                        543469
DoS Hulk                      230124
PortScan                      158804
DDoS                          128025
DoS GoldenEye                  10293
FTP-Patator                     7935
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1956
Web Attack - Brute Force        1507
Web Attack - XSS                 652
Infiltration                      36
Web Attack - Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64


In [27]:
# Temizlik sonrası numeric kolonları yeniden al
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Hangi sütunlarda %50'den fazla sıfır var?
zero_pct = (df[numeric_cols] == 0).mean() * 100
sparse_cols = zero_pct[zero_pct > 50].sort_values(ascending=False)

print("Sıfır oranı %50'nin üzerinde olan sütunlar:")
print(sparse_cols.round(1).to_string())

Sıfır oranı %50'nin üzerinde olan sütunlar:
Bwd PSH Flags            100.0
Fwd Avg Bulk Rate        100.0
Bwd URG Flags            100.0
Bwd Avg Bulk Rate        100.0
Bwd Avg Packets/Bulk     100.0
Fwd Avg Bytes/Bulk       100.0
Bwd Avg Bytes/Bulk       100.0
Fwd Avg Packets/Bulk     100.0
Fwd URG Flags            100.0
CWE Flag Count           100.0
RST Flag Count           100.0
ECE Flag Count           100.0
Fwd PSH Flags             96.2
SYN Flag Count            96.2
Active Std                95.7
Idle Std                  94.0
URG Flag Count            93.6
FIN Flag Count            93.5
Active Min                74.6
Active Max                74.6
Active Mean               74.6
Idle Max                  73.9
Idle Mean                 73.9
Idle Min                  73.9
PSH Flag Count            64.3
Bwd Packet Length Std     63.9
Bwd IAT Std               63.9
ACK Flag Count            63.0
Min Packet Length         59.9
Bwd Packet Length Min     59.5
Fwd Packet Length Std     

In [28]:
# Tamamen sıfır olan sütunlar hiçbir bilgi taşımıyor → drop
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols_to_drop = [col for col in numeric_cols if (df[col] == 0).all()]

print(f"Düşürülen sütunlar ({len(cols_to_drop)} adet):")
for c in cols_to_drop:
    print(f"  - {c}")

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nKalan sütun sayısı: {len(df.columns)}")

Düşürülen sütunlar (8 adet):
  - Bwd PSH Flags
  - Bwd URG Flags
  - Fwd Avg Bytes/Bulk
  - Fwd Avg Packets/Bulk
  - Fwd Avg Bulk Rate
  - Bwd Avg Bytes/Bulk
  - Bwd Avg Packets/Bulk
  - Bwd Avg Bulk Rate

Kalan sütun sayısı: 72


In [29]:
output_path = '../data/csv/cleaned_dataset.csv'
df.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024*1024)
print(f"✓ Final dataset kaydedildi")
print(f"  Satır: {len(df):,}")
print(f"  Sütun: {len(df.columns)}")
print(f"  Boyut: {size_mb:.1f} MB")

✓ Final dataset kaydedildi
  Satır: 1,100,025
  Sütun: 72
  Boyut: 398.1 MB
